<a href="https://colab.research.google.com/github/zeinab29/RAG_-/blob/main/RAG_%D8%A7%D9%84%D8%B5%D9%84%D8%A7%D8%A9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upload the pdf file


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 500_مسألة_في_الصلاة.pdf to 500_مسألة_في_الصلاة.pdf


# Text extraction

In [ ]:
!pip install pypdf sentence-transformers faiss-cpu groq -q

from pypdf import PdfReader

reader = PdfReader("500_مسألة_في_الصلاة.pdf")
text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print(len(text))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 13.0 MB/s eta 0:00:00
220681


In [ ]:
import re
text = re.sub(r'\n+', ' ', text)
text = re.sub(r'\s+', ' ', text)

# Splitting the text

In [ ]:

def splittingText(text, chunk_size = 200 , overlap = 50):#الـ overlap بيخلي كل chunk ياخد شوية من نهاية اللي قبله، عشان مفيش معلومة تتقطع في النص في النص بين قطعتين.
  chunks = []
  start = 0
  while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start = end - overlap
  return chunks

chunks = splittingText(text)
print(len(chunks))



1421


In [ ]:
#embeddings
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(chunks, show_progress_bar=True)
print(embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

(1421, 384)


In [ ]:
import faiss
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

#Search function
def Search(query, k=3):
  query_embedding = model.encode([query])
  distances, indices = index.search(query_embedding, k)
  return [chunks[i] for i in indices[0]]


In [ ]:
Search('ما هو حكم الصلاة؟')

[' وبهذا قال الجمهور من أهل اللغة وغيرهم من أهل التحقيق 3س : ما تعريف الصلاة في الاصطلاح ؟ 3 ج : التعبدُّ لله تعالى بأقوال وأفعال معلومة مُفتتحة بالتكبير، مختتمة بالتسليم،مع النية، بشرائط مخصوصة. 4س : م',
 ' . 500 )(مسألة في الصلاة 1س : ما تعريف الصلاة لغة ؟ 1: ج : الدعاء . أي: ادع لهم.}وَصَل ِّ عَلَيْهِّمْ { :لىوالدليل على ذلك : قول الله تعا وقال صلى الله عليه وسلم: إذا دعى أحدكم فليجب، فإن كان صائمًا ف',
 ' صلاة تفعل ما بين صلاة العشاء وطلوع الفجر، تختم بها صلاة لليل، وسميت بذلك لأنها تصلى وترًا ركعة واحدة أو ثلاثًً أو أكثر ولا تكون شفعًا. 212س : ما حُكْــــم الوتـــــر ؟ 212 في ج : تنازع العلماء وجوبه،']

In [ ]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("Groq_api_key"))

def ask(question):
    context = "\n".join(Search(question))
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": f"جاوب بس من المعلومات دي، ولو مش لاقي إجابة قول مش متأكد:\n{context}"},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

print(ask("إيه حكم من ترك الصلاة عمدًا؟"))

حكم من ترك الصّلاة عمداً هو إثم كبير. وبحسب ما ورد في المعلومة التي لديك:

* **الآراء الفقهية**  
  * **الرأي الغالب (الجمهور)**: يفرض على من ترك الصّلاة – حتى لو كان ذلك بنية الإهمال أو العناد – أن يقضى كلّ الصلوات الفريضة التي فاته.  
  * **الرأي الأقلية (المذكور في النص)**: يبيّن أن من أضاع الصّلاة وقتاً من عمره ثم تَوب إلى الله واستقام على دينه لا يَصَلِّي ما فاته، أي لا يُلْزَمُ بقضاء الصلوات الفائتة، سواء اعتُبر ذلك كفراً عند تركها أو لا.

إذن من ترك الصّلاة عمداً يُعَدّ مُذنبًا، وتختلف الفتاوى فيما إذا كان عليه قضاء الصلوات الفائتة أم لا؛ الأغلبية تشترط القضاء، بينما القليل تقول إنّ التوبة تُعفيه من ذلك.


In [ ]:
print(ask("إيه هي سجدة السهو؟"))

سجدة السهو هي السجدتان اللتان يُضيفهما المُصَلِّي إذا أدرك أنه نسي ركعةً أو فَعلًا من أركان الصلاة بعد أن انتهى منها. تُؤدَّى كالسجود العادي: يَكْبُرُ في كل سجدتين (عند كل خفض ورفع)، ثم يجلس بينهما أو بعدهما، ثم يُسَلِّم. وهذا ما يُستند إليه في الحديث المذكور بأن «السجدتين في الركعة تُامًا يكبر عند كل خفض ورفع ثم يُسَلِّم، سواء كان السجود قبل التسليم أو بعده».
